## Mixed Precision Training

Using two different numeric precisions during training at the same time — fast, lower-precision numbers for most calculations, and high-precision numbers only where accuracy actually matters.
Typically gives 2-3x training speedup with no meaningful loss in model quality. Standard practice in serious model training today.

## How it actually works

**The two number formats involved:**
- **FP32 (32-bit floating point)** — the traditional default. High precision, more memory per number, slower to compute.
- **FP16 (16-bit floating point)** — half the memory, faster to compute on modern GPU hardware, but represents numbers with less precision.

**Why not just use FP16 everywhere — the precision tradeoff:**
FP16 cannot represent very small or very large numbers as precisely as FP32.
During training, some values — particularly very small gradients — can shrink to zero
in FP16 when they would still be meaningfully nonzero in FP32. This loses information
exactly where precision matters most: in the weight update step.

**The mixed precision solution — using both strategically:**
- **Forward pass** (running input through the model to get a prediction) — done in FP16. Fast.
- **Gradient calculation** (figuring out how much each weight contributed to the error) — done in FP16. Fast.
- **Weight updates** (actually adjusting the weights) — done in FP32. Precise.
- **Master copy of the weights** — always kept in FP32. This is the authoritative, accurate version of the model.

The model effectively runs FAST in FP16 for the bulk of the computation,
but protects the most precision-sensitive step — updating the weights — by keeping it in FP32.

**Why this is faster — three separate reasons:**
1. **Memory** — FP16 numbers take half the space of FP32. The same GPU can hold a bigger batch,
   or a bigger model, in the same amount of memory.
2. **Compute speed** — Modern GPUs have specialised hardware (often called Tensor Cores)
   that process FP16 operations significantly faster than FP32 — commonly 2 to 8 times faster.
3. **Memory bandwidth** — Moving FP16 data between GPU memory and the compute units
   takes roughly half the time compared to moving the same data in FP32.

## Where it breaks or gets dangerous

**Gradient underflow — the main failure mode:**
Some gradients during training are extremely small. In FP16, very small numbers
can round down to exactly zero, losing the information entirely.
If this happens for an important weight, that weight effectively stops learning
for that step, even though in FP32 the gradient was still meaningfully nonzero.

**The standard fix — loss scaling:**
Before converting to FP16, the loss value is deliberately multiplied by a large number
(say, 1024), which scales up all the gradients proportionally, pushing small values
away from the zero-underflow zone. After the gradients are calculated, they are divided
back down by the same factor before the weight update. This preserves the small gradient information
that would otherwise have been lost to FP16's limited range.

**Not free — requires careful implementation:**
Getting mixed precision training right is not just "flip a switch and everything is faster."
Loss scaling needs to be tuned or handled automatically by the training framework.
Done carelessly, you can get silent training instability — loss curves that look unstable
or diverge, for reasons that trace back to precision handling rather than the model
or data itself (connecting back to notebook on debugging unstable training loss).

**This is a training-time concept, not an inference-time one — important distinction:**
Mixed precision training is about how a model is TRAINED. When you use Claude via the API,
you are doing inference — sending a prompt, getting a response — not training.
You are not making decisions about FP16 vs FP32 in your own projects, because Anthropic
handles all of that on their infrastructure when training the underlying model.
This concept matters for your interview readiness and your understanding of how frontier
models are built — but it is not something you configure in your day-to-day API usage.

**Where it would become directly relevant to you:**
If you ever fine-tune a model locally — using LoRA/QLoRA, as discussed in the catastrophic
forgetting notebook — mixed precision settings become something you would actually configure,
because you would be doing real training, even if only updating a small set of parameters.

In [ ]:
# Mixed Precision Training — conceptual illustration of FP16 underflow and loss scaling

import numpy as np

def simulate_fp16_rounding(value, fp16_min_normal=6.1e-5):
    """
    Rough simulation of FP16 underflow.
    Real FP16 has its own exact rules, but this captures the core idea:
    very small numbers can effectively become zero.
    """
    if abs(value) < fp16_min_normal:
        return 0.0
    return value

def train_step_no_loss_scaling(gradients):
    """Simulate applying FP16 rounding directly to small gradients."""
    return [simulate_fp16_rounding(g) for g in gradients]

def train_step_with_loss_scaling(gradients, scale_factor=1024):
    """
    Simulate loss scaling: scale gradients up before FP16 rounding,
    then scale back down afterward — preserving small-value information.
    """
    scaled_up = [g * scale_factor for g in gradients]
    rounded = [simulate_fp16_rounding(g) for g in scaled_up]
    scaled_back_down = [g / scale_factor for g in rounded]
    return scaled_back_down

# Simulate a set of gradients, including some very small ones
gradients = [0.5, 0.001, 0.00008, 0.000003, 0.02, 0.0000009]

print(f"{'Original Gradient':<20}{'No Scaling (FP16)':<22}{'With Loss Scaling':<22}")
print("-" * 64)

no_scaling_result = train_step_no_loss_scaling(gradients)
with_scaling_result = train_step_with_loss_scaling(gradients)

for orig, no_scale, with_scale in zip(gradients, no_scaling_result, with_scaling_result):
    print(f"{orig:<20.8f}{no_scale:<22.8f}{with_scale:<22.8f}")

print()
print("Notice: without loss scaling, the smallest gradients round down to exactly 0.0 —")
print("that weight effectively stops learning from this step entirely.")
print()
print("With loss scaling, those same small gradients survive the FP16 conversion")
print("because they were scaled up above the rounding threshold before conversion,")
print("then scaled back down afterward to their correct, original magnitude.")

## My D365 analogy

Mixed precision training is like running daily transactional reporting using rounded,
summary-level figures for speed — but keeping the full, unrounded detail in the underlying
ledger for the actual period-end close and audit.

Day to day, finance teams do not need ten decimal places of precision to glance at
cash flow trends or daily sales summaries. Rounded, fast, lightweight figures are good enough
and let people work quickly without waiting on heavy, precise calculations for every glance.

But at period-end close — the moment that actually determines the official numbers —
you fall back to the full-precision ledger. That is where rounding errors would actually matter,
compounding into real discrepancies if you used summary-level figures for the official close.

Mixed precision training applies the same logic to a neural network:
use the fast, lower-precision version for the bulk of repetitive calculation,
but protect the one step where precision loss would actually corrupt the result —
the weight update, the "official record" of what the model has learned.

Speed everywhere it is safe to be fast. Precision exactly where it is not.

## LinkedIn post idea

Training a large model with full precision everywhere is like insisting on ten decimal places
for every number in a daily operational report. Technically more accurate. Mostly unnecessary.
And very, very slow.

Modern model training does something more deliberate: use fast, lower-precision numbers
for most of the heavy lifting, and reserve full precision only for the one step
where losing accuracy would actually corrupt the result — updating the model's weights.

This single design choice — called mixed precision training — typically cuts training time
by half or more, with no meaningful drop in the quality of the final model.

What stuck with me learning this: the solution was not "use less precision" or "use more precision."
It was deciding, deliberately, exactly where precision matters and where it does not —
and treating those two zones completely differently.

That is not really a deep learning insight. It is a resource allocation insight
wearing a deep learning costume. I have made the same call a hundred times
deciding which D365 configurations need rigorous, careful change control,
and which ones can move fast without much ceremony.